# Revenue Report Generator
Builds monthly and combined invoice/credit summaries by `a/c_no` from `Supermarket_dpts_vans_inst.xlsx`.

In [117]:
import numpy as np
import pandas as pd
import openpyxl

In [118]:
# --- CONFIG ---
INPUT_PATH = "data/Supermarket_dpts_vans_inst.xlsx"
OUTPUT_PATH = "reports/powerbi_revenue_report.xlsx"

MONTH_ORDER = ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug",
               "sep", "oct", "nov", "dec"]

## Load data

In [119]:
df_invoice = pd.read_excel(INPUT_PATH, sheet_name="Invoices")
df_credit = pd.read_excel(INPUT_PATH, sheet_name="Credits")

## Cleaning / preprocessing

In [120]:
def clean(df: pd.DataFrame) -> pd.DataFrame:
    df = df.drop(columns=["no"], errors="ignore").copy()
    df.columns = (
        df.columns.str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("(", "", regex=False)
        .str.replace(")", "", regex=False)
        .str.replace(".", "", regex=False)
    )
    df["date"] = pd.to_datetime(df["date"])
    df["month_name"] = df["date"].dt.strftime("%b").str.lower()
    # a/c_no as a clean string key so invoice/credit sides always match
    df["a/c_no"] = df["a/c_no"].astype(str).str.strip()
    return df


df_invoice = clean(df_invoice)
df_credit = clean(df_credit)
df_credit["amount"] = df_credit["amount"].abs()

df_invoice.head()

,date,no,invoice_no,a/c_no,customers,category,route,qty,total_amt,vat_amt,extra_cts,amount,month_name
0,2026-01-01,1964260001,855312,1964,MAGUNAS SUPER STORES (K) LTD JOGOO ROAD (1964),DSUPMK,DAKAR SALES,77.0,4487.0,0.0,0.0,4487.0,jan
1,2026-01-01,1501260001,855314,1501,MAJID AL FUTTAIM - THE JUNCTION (1501),DSUPMK,WAITHAKA 1 ROUTE,140.0,10219.0,0.0,0.0,10219.0,jan
2,2026-01-01,1998260001,855317,1998,KAHAWA WEST,DIST,KAHAWA WEST 1 ROUTE,1632.0,82872.0,0.0,666.0,83538.0,jan
3,2026-01-01,2146260001,855319,2146,SKY MART ENTERPRISES LIMITED - KAWANGWARE (2146),DSUPMK,WAITHAKA 1 ROUTE,70.0,5294.0,0.0,70.0,5364.0,jan
4,2026-01-01,1304260001,855320,1304,NAIVAS SUPERMARKET LTD- GREENHOUSE NGONG RD (1...,DSUPMK,WAITHAKA 1 ROUTE,47.0,3589.0,0.0,0.0,3589.0,jan


In [121]:
# a/c_no -> customer name lookup (first non-null match wins)
cust_lookup = (
    pd.concat([df_invoice[["a/c_no", "customers"]], df_credit[["a/c_no", "customers"]]])
    .dropna(subset=["a/c_no"])
    .drop_duplicates(subset="a/c_no", keep="first")
    .set_index("a/c_no")["customers"]
)

# category and route are attributes of the account too - keep first non-null match
category_lookup = (
    pd.concat([df_invoice[["a/c_no", "category"]], df_credit[["a/c_no", "category"]]])
    .dropna(subset=["a/c_no"])
    .drop_duplicates(subset="a/c_no", keep="first")
    .set_index("a/c_no")["category"]
)

route_lookup = (
    pd.concat([df_invoice[["a/c_no", "route"]], df_credit[["a/c_no", "route"]]])
    .dropna(subset=["a/c_no"])
    .drop_duplicates(subset="a/c_no", keep="first")
    .set_index("a/c_no")["route"]
)

months_present = [m for m in MONTH_ORDER
                   if m in set(df_invoice["month_name"]) | set(df_credit["month_name"])]
months_present

['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep']

## Per-month summaries (one sheet per month)
For each `a/c_no`: total invoice amount, total credit amount, and `pct_return = credit_amt / invoice_amt`.

In [122]:
monthly_summaries = {}

for m in months_present:
    inv_sum = (
        df_invoice.loc[df_invoice["month_name"] == m]
        .groupby("a/c_no")["amount"].sum()
        .rename("invoice_amt")
    )
    cred_sum = (
        df_credit.loc[df_credit["month_name"] == m]
        .groupby("a/c_no")["amount"].sum()
        .rename("credit_amt")
    )

    summary = pd.concat([inv_sum, cred_sum], axis=1).fillna(0).reset_index()
    summary["pct_return"] = np.where(
        summary["invoice_amt"] != 0,
        (summary["credit_amt"] / summary["invoice_amt"]),
        0,
    ).round(3)
    summary["customers"] = summary["a/c_no"].map(cust_lookup)
    summary["category"] = summary["a/c_no"].map(category_lookup)
    summary["route"] = summary["a/c_no"].map(route_lookup)
    summary = summary[["a/c_no", "customers", "category", "route", "invoice_amt", "credit_amt", "pct_return"]]
    summary = summary.sort_values("a/c_no").reset_index(drop=True)

    monthly_summaries[m] = summary

monthly_summaries[months_present[0]].head()

,a/c_no,customers,category,route,invoice_amt,credit_amt,pct_return
0,100,MAJENGO,DIST,DAKAR SALES,1474914.000,40335.000,0.027
1,1000,PIPELINE NRB,DIST,MULOLONGO ROUTE,1584862.500,17541.000,0.011
2,1003,NGOINGWA THIKA,DIST,KIANDUTU ROUTE,1094417.580,77531.500,0.071
3,1008,NAIVAS SUPERMARKET -NAIVASHA KUBWA (1008),DSUPMK,NAIVASHA SALES,284345.994,10931.994,0.038
4,1009,ROYSAMBU -NRB,DIST,KAHAWA WEST 1 ROUTE,1507174.500,123806.500,0.082


## Combined monthly sheet (wide format)
One row per `a/c_no`, with `<mon>_invoice`, `<mon>_credit`, `<mon>_pct_rtn` columns for every month present.

In [123]:
all_accounts = sorted(set(df_invoice["a/c_no"]) | set(df_credit["a/c_no"]))
combined = pd.DataFrame({"a/c_no": all_accounts})
combined["customers"] = combined["a/c_no"].map(cust_lookup)
combined["category"] = combined["a/c_no"].map(category_lookup)
combined["route"] = combined["a/c_no"].map(route_lookup)

for m in months_present:
    s = monthly_summaries[m].set_index("a/c_no")
    combined[f"{m}_invoice"] = combined["a/c_no"].map(s["invoice_amt"]).fillna(0)
    combined[f"{m}_credit"] = combined["a/c_no"].map(s["credit_amt"]).fillna(0)
    combined[f"{m}_pct_rtn"] = np.where(
        combined[f"{m}_invoice"] != 0,
        (combined[f"{m}_credit"] / combined[f"{m}_invoice"]),
        0,
    ).round(3)

combined.head()

,a/c_no,customers,category,route,jan_invoice,jan_credit,jan_pct_rtn,feb_invoice,feb_credit,feb_pct_rtn,...,jun_pct_rtn,jul_invoice,jul_credit,jul_pct_rtn,aug_invoice,aug_credit,aug_pct_rtn,sep_invoice,sep_credit,sep_pct_rtn
0,100,MAJENGO,DIST,DAKAR SALES,1474914.000,40335.000,0.027,1259176.500,4026.500,0.003,...,0.079,1389209.500,25273.50,0.018,1375825.000,10137.500,0.007,1379400.000,54422.500,0.039
1,1000,PIPELINE NRB,DIST,MULOLONGO ROUTE,1584862.500,17541.000,0.011,1541522.500,17307.000,0.011,...,0.007,1584602.000,15566.00,0.010,1886387.505,134736.000,0.071,1887462.555,49871.030,0.026
2,1003,NGOINGWA THIKA,DIST,KIANDUTU ROUTE,1094417.580,77531.500,0.071,912614.460,43949.520,0.048,...,0.012,1279823.500,68265.50,0.053,1043132.510,22219.000,0.021,1109437.595,17988.022,0.016
3,1008,NAIVAS SUPERMARKET -NAIVASHA KUBWA (1008),DSUPMK,NAIVASHA SALES,284345.994,10931.994,0.038,278210.988,27944.004,0.100,...,0.102,342784.916,13240.50,0.039,392895.446,14764.506,0.038,370128.982,22476.498,0.061
4,1009,ROYSAMBU -NRB,DIST,KAHAWA WEST 1 ROUTE,1507174.500,123806.500,0.082,1232652.000,18372.000,0.015,...,0.025,1260474.000,18916.25,0.015,1312862.505,18070.500,0.014,1300610.215,19114.500,0.015


## Grand total summary (all months combined, per account)
The `Brand_Summary` sheet combines branches for major supermarket chains into one total per brand.

In [124]:
total_invoice = df_invoice.groupby("a/c_no")["amount"].sum().rename("total_invoice")
total_credit = df_credit.groupby("a/c_no")["amount"].sum().rename("total_credit")

total_summary = pd.concat([total_invoice, total_credit], axis=1).fillna(0).reset_index()
total_summary["customers"] = total_summary["a/c_no"].map(cust_lookup)
total_summary["category"] = total_summary["a/c_no"].map(category_lookup)
total_summary["route"] = total_summary["a/c_no"].map(route_lookup)
total_summary["total_pct_return"] = np.where(
    total_summary["total_invoice"] != 0,
    (total_summary["total_credit"] / total_summary["total_invoice"]),
    0,
).round(3)
total_summary = total_summary[
    ["a/c_no", "customers", "category", "route", "total_invoice", "total_credit", "total_pct_return"]
]
total_summary["a/c_no"] = pd.to_numeric(total_summary["a/c_no"])
total_summary = total_summary.sort_values("a/c_no").reset_index(drop=True)

customer_names = total_summary["customers"].astype("string")
normalized_names = (
    customer_names.fillna("").str.upper()
    .str.replace(r"\s*\(\s*\d+\s*\)\s*$", "", regex=True)
    .str.replace(r"[^A-Z0-9]+", " ", regex=True)
    .str.strip()
)
name_prefixes = normalized_names.str.extract(r"^([A-Z0-9]+)", expand=False)
prefix_counts = name_prefixes.value_counts()
generic_prefixes = {"THE", "NEW", "ST", "COUNTY", "P", "KISII", "THIKA", "NYAHURURU", "MAUA", "KIKUYU"}
shared_brand_prefix = (
    name_prefixes.map(prefix_counts).gt(1)
    & ~name_prefixes.isin(generic_prefixes)
    & name_prefixes.notna()
)
brand_summary_source = total_summary.copy()
brand_summary_source["_group_key"] = normalized_names
brand_summary_source.loc[shared_brand_prefix, "_group_key"] = (
    "brand:" + name_prefixes.loc[shared_brand_prefix]
)
missing_customer = normalized_names.eq("")
brand_summary_source.loc[missing_customer, "_group_key"] = (
    "account:" + brand_summary_source.loc[missing_customer, "a/c_no"].astype(str)
)
brand_summary_source = brand_summary_source.sort_values("a/c_no")
brand_summary = (
    brand_summary_source.groupby("_group_key", as_index=False, sort=False)
    .agg({
        "a/c_no": "first",
        "customers": "first",
        "category": "first",
        "route": "first",
        "total_invoice": "sum",
        "total_credit": "sum",
    })
)
brand_summary["total_pct_return"] = np.where(
    brand_summary["total_invoice"] != 0,
    brand_summary["total_credit"] / brand_summary["total_invoice"],
    0,
).round(3)
brand_summary = brand_summary[
    ["a/c_no", "customers", "category", "route", "total_invoice", "total_credit", "total_pct_return"]
].reset_index(drop=True)

total_summary.head()

,a/c_no,customers,category,route,total_invoice,total_credit,total_pct_return
0,55,RUIRUNGU,DIST,KAMAKWA ROUTE,7.864607e+06,118984.320,0.015
1,57,KARATINA,DIST,KARATINA ROUTE,3.280988e+07,1028785.245,0.031
2,62,RUIRU,DIST,RUIRU ROUTE,2.583214e+07,710219.580,0.027
3,63,MACHAKOS,DIST,MACHAKOS ROUTE,2.675112e+07,774055.000,0.029
4,82,KANGEMI,DIST,KANGEMI ROUTE,8.033151e+06,272763.840,0.034


## Daily total (all accounts combined)
A day-level total, not split by account: total invoiced, total credited, and pct return per day.

In [125]:
daily_invoice = df_invoice.groupby("date")["amount"].sum().rename("invoice_amt")
daily_credit = df_credit.groupby("date")["amount"].sum().rename("credit_amt")

daily_summary = pd.concat([daily_invoice, daily_credit], axis=1).fillna(0).reset_index()
daily_summary["pct_return"] = np.where(
    daily_summary["invoice_amt"] != 0,
    (daily_summary["credit_amt"] / daily_summary["invoice_amt"]),
    0,
).round(3)
daily_summary = daily_summary.sort_values("date").reset_index(drop=True)

daily_summary.head()

,date,invoice_amt,credit_amt,pct_return
0,2026-01-01,1.201862e+07,511360.382,0.043
1,2026-01-02,1.463413e+07,616006.606,0.042
2,2026-01-03,1.253877e+07,281146.981,0.022
3,2026-01-04,1.148173e+07,113561.485,0.010
4,2026-01-05,1.492134e+07,1661071.741,0.111


## Write output workbook

In [126]:
for frame in [*monthly_summaries.values(), combined, total_summary]:
    frame["a/c_no"] = pd.to_numeric(frame["a/c_no"])

with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
    for m in months_present:
        monthly_summaries[m].to_excel(writer, sheet_name=m.upper(), index=False)
    combined.to_excel(writer, sheet_name="Combined_Monthly", index=False)
    total_summary.to_excel(writer, sheet_name="Total_Summary", index=False)
    brand_summary.to_excel(writer, sheet_name="Brand_Summary", index=False)
    daily_summary.to_excel(writer, sheet_name="Daily_Summary", index=False)
    for ws in writer.book.worksheets:
        for header in ws[1]:
            if header.value and ("pct" in str(header.value).lower() or "return" in str(header.value).lower()):
                for cells in ws.iter_rows(min_row=2, min_col=header.column, max_col=header.column):
                    cells[0].number_format = "0.000"

print(f"Report saved to {OUTPUT_PATH}")
print(f"Months found: {months_present}")
print(f"Accounts: {len(all_accounts)}")

Report saved to reports/powerbi_revenue_report.xlsx
Months found: ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep']
Accounts: 997
